In [16]:
# from pynq import Overlay
# from pynq import MMIO
import numpy as np
import time

import cv2

In [17]:
def quantize_8bit(x):
    # PyTorch tensor를 numpy로 변환
    temp = x
    temp = np.floor(temp)  # Round the values ######################################################
    # 9 LSB 제거 (오른쪽 시프트 9)
    output_shifted = np.right_shift(temp.astype(np.int32), 9)
    # 8-bit signed int 범위로 클리핑 (-128 ~ 127)
    quantized = np.clip(output_shifted, -128, 127).astype(np.int8)
    return quantized

In [18]:
import numpy as np

def reshape_input_and_weights(x, weights, fc_key='fc', kernel_size=3, oc_size=10):
    # Flatten input
    input_flat = x.flatten()
    
    # Calculate padding length
    total_length = np.ceil(len(input_flat) / (kernel_size * kernel_size)) * (kernel_size * kernel_size)
    pad_length = int(total_length - len(input_flat))
    
    # Pad input
    input_padded = np.pad(input_flat, (0, pad_length), mode='constant', constant_values=0)
    
    # Reshape input
    ic = int(total_length // (kernel_size * kernel_size))
    input_reshaped = input_padded.reshape(ic, kernel_size, kernel_size)
    
    # Load weight data
    weight_data = weights[fc_key]
    
    # Initialize reshaped weights array
    weight_reshaped = np.zeros((oc_size, ic, kernel_size, kernel_size), dtype=weight_data.dtype)
    
    # Reshape weights for each output channel
    for oc in range(oc_size):
        weight_flat = weight_data[oc]
        weight_padded = np.pad(weight_flat, (0, pad_length), mode='constant', constant_values=0)
        weight_oc_reshaped = weight_padded.reshape(ic, kernel_size, kernel_size)
        weight_reshaped[oc] = weight_oc_reshaped
    
    return input_reshaped, weight_reshaped

In [19]:
def conv2d_IN_HW(i_act_tile, weight_tile, tile_oc_act_flat, quantize_8bit, kernel_h, kernel_w, input_ch, oc_range, h_range, w_range):
    stride=1
    # input_ch, tile_h_input, tile_w_input = i_act_tile.shape
    # oc_range, input_ch_w, kernel_h, kernel_w = weight_tile.shape
    # oc_range_out, h_range, w_range = tile_o_act.shape

    for toc in range(oc_range):
        for toh in range(h_range):
            h_start = toh * stride
            for tow in range(w_range):
                w_start = tow * stride
                temp = 0
                for ic in range(input_ch):
                    for kh in range(kernel_h):
                        for kw in range(kernel_w):
                            val = i_act_tile[ic, h_start + kh, w_start + kw].astype(np.float32)
                            wgt = weight_tile[toc, ic, kh, kw].astype(np.float32)
                            temp += val * wgt
                temp = quantize_8bit(temp)
                tile_oc_act_flat[toc*h_range*w_range + toh*w_range + tow] = temp
    return tile_oc_act_flat

In [20]:
def conv2d_HW(i_act, weight, stride=1, tile_h=8, tile_w=8, tile_oc=8, padding=0):
    """
    conv2d (9 for loops) with tiling.
    - i_act: shape (input_ch, input_h, input_w)
    - weight: shape (output_ch, input_ch, kernel_h, kernel_w)
    """
    input_ch, input_h, input_w = i_act.shape
    output_ch, _, kernel_h, kernel_w = weight.shape

    # 출력 feature map 크기 계산
    output_h = (input_h - kernel_h) // stride + 1
    output_w = (input_w - kernel_w) // stride + 1
    o_act = np.zeros((output_ch, output_h, output_w)).astype(np.float32)

    # 타일 단위 반복
    for oh in range(0, output_h, tile_h):
        for ow in range(0, output_w, tile_w):
            for oc in range(0, output_ch, tile_oc):
                h_range = min(tile_h, output_h - oh)
                w_range = min(tile_w, output_w - ow)
                oc_range = min(tile_oc, output_ch - oc)

                # 타일 단위로 i_act와 weight 슬라이싱
                h_start = oh * stride
                w_start = ow * stride
                    
                h_end = h_start + h_range * stride + kernel_h - 1
                w_end = w_start + w_range * stride + kernel_w - 1
                
                i_act_tile = i_act[:, h_start:h_end, w_start:w_end]
                weight_tile = weight[oc:oc + oc_range, :, :, :]

                tile_oc_act_flat = np.zeros(oc_range * h_range * w_range).astype(np.float32)
                tile_o_act = np.zeros((oc_range, h_range, w_range)).astype(np.float32)
                
                input_ch, tile_h_input, tile_w_input = i_act_tile.shape
                oc_range, input_ch_w, kernel_h, kernel_w = weight_tile.shape
                oc_range_out, h_range, w_range = tile_o_act.shape

                # convolution 연산
                tile_oc_act_flat = conv2d_IN_HW(i_act_tile, weight_tile, tile_oc_act_flat, quantize_8bit, kernel_h, kernel_w, input_ch, oc_range, h_range, w_range)
                tile_o_act = tile_oc_act_flat.reshape(oc_range, h_range, w_range)
                # print("tile_o_act shape:", tile_o_act.shape)  # Debugging output

                o_act[oc:oc + oc_range, oh:oh + h_range, ow:ow + w_range] = tile_o_act

    return o_act

In [21]:
def fc_HW(i_act, weight, stride=1, tile_h=3, tile_w=3, tile_oc=1, padding=0):
    # 입력 데이터를 flatten: (12800,)
    input_flat = i_act.flatten()

    # 필요한 패딩 길이 계산: 12800을 72로 나누어 떨어지게 하기 위해 ceil(12800/72)*72 = 12816
    total_length = np.ceil(len(input_flat) / 72) * 72
    pad_length = int(total_length - len(input_flat))  # 16

    # 입력 데이터 패딩: (12807,)
    input_padded = np.pad(input_flat, (0, pad_length), mode='constant', constant_values=0)

    # reshape: (1423 -> 1424, 3, 3)         # IC=1423 -> 1424, H=3, W=3
    ic = int(total_length // 9)     # 1423 -> 1424
    input_reshaped = input_padded.reshape(ic, 3, 3)

    print("INPUT shaping 이후 data 형태:", input_reshaped.shape)

    # 가중치 데이터 로드: (10, 12800)
    weight_data = weight

    # 결과 가중치 배열 초기화: (10, 1423 -> 1424, 3, 3)  # OC=10, IC=1423 -> 1424, H=3, W=3
    weight_reshaped = np.zeros((10, ic, 3, 3), dtype=weight_data.dtype)

    print("WEIGHT shaping 이후 data 형태:", weight_reshaped.shape)

    # 각 OC에 대해 패딩 및 reshape
    for oc in range(10):
        weight_flat = weight_data[oc]
        weight_padded = np.pad(weight_flat, (0, pad_length), mode='constant', constant_values=0)
        weight_oc_reshaped = weight_padded.reshape(ic, 3, 3)
        weight_reshaped[oc] = weight_oc_reshaped
    
    """
    Implement FC with conv2d (9 for loops) with tiling.
    - i_act: shape (input_ch, input_h, input_w)
    - weight: shape (output_ch, input_ch, kernel_h, kernel_w)
    """
    input_ch, input_h, input_w = input_reshaped.shape
    output_ch, _, kernel_h, kernel_w = weight_reshaped.shape
    
    # 출력 feature map 크기 계산
    output_h = (input_h - kernel_h) // stride + 1
    output_w = (input_w - kernel_w) // stride + 1
    o_act = np.zeros((output_ch, output_h, output_w))

    # 타일 단위 반복
    for oh in range(0, output_h, tile_h):
        for ow in range(0, output_w, tile_w):
            for oc in range(0, output_ch, tile_oc):
                h_range = min(tile_h, output_h - oh)
                w_range = min(tile_w, output_w - ow)
                oc_range = min(tile_oc, output_ch - oc)

                # 타일 단위로 i_act와 weight 슬라이싱
                h_start = oh * stride
                w_start = ow * stride
                    
                h_end = h_start + h_range * stride + kernel_h - 1
                w_end = w_start + w_range * stride + kernel_w - 1
                
                i_act_tile = input_reshaped[:, h_start:h_end, w_start:w_end]
                weight_tile = weight_reshaped[oc:oc + oc_range, :, :, :]

                tile_o_act = np.zeros((oc_range, h_range, w_range))
                
                input_ch, tile_h_input, tile_w_input = i_act_tile.shape
                oc_range, input_ch_w, kernel_h, kernel_w = weight_tile.shape
                oc_range_out, h_range, w_range = tile_o_act.shape
                
                # convolution 연산
                tile_o_act = conv2d_IN_HW(i_act_tile, weight_tile, tile_o_act, quantize_8bit, kernel_h, kernel_w, input_ch, oc_range, h_range, w_range)

                o_act[oc:oc + oc_range, oh:oh + h_range, ow:ow + w_range] = tile_o_act

    return o_act

In [22]:
def leaky_relu_loop(input_arr, alpha=0.25):
    """
    Leaky ReLU.
    - input_arr: shape (ch, h, w)
    """
    output = np.zeros_like(input_arr).astype(np.float32)
    ch, h, w = input_arr.shape
    for c in range(ch):
        for i in range(h):
            for j in range(w):
                x = input_arr[c, i, j].astype(np.float32)
                output[c, i, j] = x if x > 0 else alpha * x
    return output

In [23]:
def maxpool_loop(input_arr, pool_size=2, stride=2):
    """
    Max pooling
    - input_arr: shape (ch, h, w)
    """
    ch, h, w = input_arr.shape
    out_h = (h - pool_size) // stride + 1
    out_w = (w - pool_size) // stride + 1
    output = np.zeros((ch, out_h, out_w)).astype(np.float32)
    for c in range(ch):
        for oh in range(out_h):
            h_start = oh * stride
            for ow in range(out_w):
                w_start = ow * stride
                max_val = -np.inf
                for ph in range(pool_size):
                    for pw in range(pool_size):
                        val = input_arr[c, h_start + ph, w_start + pw].astype(np.float32)
                        if val > max_val:
                            max_val = val
                output[c, oh, ow] = max_val
    return output

In [24]:
class NPUDriver:
    #####################################################
    # def __init__(self, bitfile_path):
    #     self.hw = Overlay(bitfile_path)
    #     self.csr = self.hw.csr_0.mmio.array
    #     self.imem = self.hw.INPUT_MEM.mmio.array
    #     self.omem = self.hw.OUT_MEM.mmio.array
    #     self.wmem = self.hw.WEIGHT_MEM.mmio.array
    #     self.result = []
    #####################################################
    def __init__(self):
        # 가상 CSR 인터페이스 (실제 HW: memory mapped) 및 가상 메모리
        self.csr = [0] * 8
        self.imem = None  # tiled input
        self.wmem = None  # tiled weight
        self.omem = None
     #####################################################

        # 타일 크기
        self.tile_h = 8
        self.tile_w = 8
        self.tile_oc = 8
        
        # Leaky ReLU alpha
        self.relu_alpha = 0.25

    def write_csr(self, address, value):
        """CSR write 시뮬레이션"""
        print("Addr: ", address, "Value: ", value)
        print(f"CSR Write: Addr {hex(address)}, Value {value}")
        self.csr[address] = value

    def config_layer(self, kw , kh, ic, input_w, input_h, oc):
        """레이어 config 설정"""
        # 공통 config
        self.write_csr(0x02, kw)
        self.write_csr(0x03, kh)
        self.write_csr(0x04, ic)
        self.write_csr(0x05, input_w)
        self.write_csr(0x06, input_h)
        self.write_csr(0x07, oc)

    def load_data(self, input_data, weight_data):
        """데이터 로드: WMEM (weight), IMEM (input) -> flatten 시켜서"""
        self.imem = input_data.astype(np.float32).ravel()
        self.wmem = weight_data.astype(np.float32).ravel()
        print(f"IMEM Load: {self.imem.shape}")
        print(f"WMEM Load: {self.wmem.shape}")

    def get_data(self, num_elements):
        """데이터 가져오기: OMEM (output)"""
        return self.omem[0:num_elements]

    def start_npu(self, input_data, weight_data, kw , kh, ic, input_w, input_h, oc):
        """데이터 로드: WMEM (weight), IMEM (input)"""
        self.load_data(input_data, weight_data)
        """레이어 config 설정"""
        self.config_layer(kw , kh, ic, input_w, input_h, oc)
        """Start Signal pulse"""
        self.write_csr(0x01, 1)
        print("NPU Started")
        
        while True:
            if (self.csr[0] == 1):
                print("NPU Done")
                break
        return self.get_data((input_w-2)*(input_h-2)*oc)
        # self.result = self.get_data((input_w-2)*(input_h-2)*oc)

    def run_conv_2d(self, input_data, weight_data, tile_h=8, tile_w=8, tile_oc=8):
        """
        conv2d (9 for loops) with tiling.
        - i_act: shape (input_ch, input_h, input_w)
        - weight: shape (output_ch, input_ch, kernel_h, kernel_w)
        """
        stride = 1
        input_ch, input_h, input_w = input_data.shape
        output_ch, _, kernel_h, kernel_w = weight_data.shape

        # 출력 feature map 크기 계산
        output_h = (input_h - kernel_h) // stride + 1
        output_w = (input_w - kernel_w) // stride + 1
        o_act = np.zeros((output_ch, output_h, output_w)).astype(np.float32)

        # 타일 단위 반복
        for oh in range(0, output_h, tile_h):
            for ow in range(0, output_w, tile_w):
                for oc in range(0, output_ch, tile_oc):
                    h_range = min(tile_h, output_h - oh)
                    w_range = min(tile_w, output_w - ow)
                    oc_range = min(tile_oc, output_ch - oc)

                    # 타일 단위로 i_act와 weight 슬라이싱
                    h_start = oh * stride
                    w_start = ow * stride
                        
                    h_end = h_start + h_range * stride + kernel_h - 1
                    w_end = w_start + w_range * stride + kernel_w - 1
                    
                    i_act_tile = input_data[:, h_start:h_end, w_start:w_end]
                    weight_tile = weight_data[oc:oc + oc_range, :, :, :]

                    tile_oc_act_flat = np.zeros(oc_range * h_range * w_range).astype(np.float32)
                    tile_o_act = np.zeros((oc_range, h_range, w_range)).astype(np.float32)
                    
                    input_ch, tile_h_input, tile_w_input = i_act_tile.shape
                    oc_range, input_ch_w, kernel_h, kernel_w = weight_tile.shape
                    oc_range_out, h_range, w_range = tile_o_act.shape

                    ########################################################################################################################
                    # convolution 연산
                    # tile_oc_act_flat = self.start_npu(i_act_tile, weight_tile, kernel_w, kernel_h, input_ch, tile_w_input, tile_h_input, oc_range)
                    ########################################################################################################################
                    # convolution 연산
                    tile_oc_act_flat = conv2d_IN_HW(i_act_tile, weight_tile, tile_oc_act_flat, quantize_8bit, kernel_h, kernel_w, input_ch, oc_range, h_range, w_range)
                    ########################################################################################################################
                    tile_o_act = tile_oc_act_flat.reshape(oc_range, h_range, w_range)
                    
                    o_act[oc:oc + oc_range, oh:oh + h_range, ow:ow + w_range] = tile_o_act

        return o_act

    def run_fc_2d(self, i_act, weight, stride=1, tile_h=3, tile_w=3, tile_oc=1, padding=0):
        """
        Implement FC with conv2d (9 for loops) with tiling.
        - i_act: shape (input_ch, input_h, input_w)
        - weight: shape (output_ch, input_ch, kernel_h, kernel_w)
        """
        input_ch, input_h, input_w = i_act.shape
        output_ch, _, kernel_h, kernel_w = weight.shape

        # 출력 feature map 크기 계산
        output_h = (input_h - kernel_h) // stride + 1
        output_w = (input_w - kernel_w) // stride + 1
        o_act = np.zeros((output_ch, output_h, output_w))

        # 타일 단위 반복
        for oh in range(0, output_h, tile_h):
            for ow in range(0, output_w, tile_w):
                for oc in range(0, output_ch, tile_oc):
                    h_range = min(tile_h, output_h - oh)
                    w_range = min(tile_w, output_w - ow)
                    oc_range = min(tile_oc, output_ch - oc)

                    # 타일 단위로 i_act와 weight 슬라이싱
                    h_start = oh * stride
                    w_start = ow * stride
                        
                    h_end = h_start + h_range * stride + kernel_h - 1
                    w_end = w_start + w_range * stride + kernel_w - 1

                    i_act_tile = i_act[:, h_start:h_end, w_start:w_end]
                    weight_tile = weight[oc:oc + oc_range, :, :, :]

                    tile_oc_act_flat = np.zeros(oc_range * h_range * w_range).astype(np.float32)
                    tile_o_act = np.zeros((oc_range, h_range, w_range))
                    
                    input_ch, tile_h_input, tile_w_input = i_act_tile.shape
                    oc_range, input_ch_w, kernel_h, kernel_w = weight_tile.shape
                    oc_range_out, h_range, w_range = tile_o_act.shape
                    
                    ########################################################################################################################
                    # convolution 연산
                    # tile_oc_act_flat = self.start_npu(i_act_tile, weight_tile, kernel_w, kernel_h, input_ch, tile_w_input, tile_h_input, oc_range)
                    ########################################################################################################################
                    # convolution 연산
                    tile_oc_act_flat = conv2d_IN_HW(i_act_tile, weight_tile, tile_oc_act_flat, quantize_8bit, kernel_h, kernel_w, input_ch, oc_range, h_range, w_range)
                    ########################################################################################################################
                    tile_o_act = tile_oc_act_flat.reshape(oc_range, h_range, w_range)

                    o_act[oc:oc + oc_range, oh:oh + h_range, ow:ow + w_range] = tile_o_act

        return o_act


In [25]:
driver = NPUDriver()

In [26]:
input_image = np.load('npy_files/input.npy')
weights = {
    'conv1': np.load('npy_files/layer1_0_weight.npy'),
    'conv2': np.load('npy_files/layer2_0_weight.npy'),
    'conv3': np.load('npy_files/layer3_0_weight.npy'),
    'conv4': np.load('npy_files/layer4_0_weight.npy'),
    'fc': np.load('npy_files/fc1_weight.npy')
}

In [27]:
def npu_golden_model_HW(input_image, weights):
    """
    Golden Model을 사용한 전체 CNN Forward Propagation.
    - input_image: 형태 (1, 28, 28)의 np.array
    - weights: dict, key 'conv1' (16,1,3,3), 'conv2' (64,16,3,3), 'conv3' (128,64,3,3), 'conv4' (128,128,3,3), 'fc' (12800,10)
    """
    # Conv1 + Leaky ReLU
    x = driver.run_conv_2d(input_image, weights['conv1'], tile_h=8, tile_w=8, tile_oc=8)
    print("Conv1 output shape:", x.shape)  # Debugging output
    x = leaky_relu_loop(x)

    # Conv2 + Leaky ReLU
    x = driver.run_conv_2d(x, weights['conv2'], tile_h=8, tile_w=8, tile_oc=8)
    print("Conv2 output shape:", x.shape)  # Debugging output
    x = leaky_relu_loop(x)

    # Conv3 + Leaky ReLU
    x = driver.run_conv_2d(x, weights['conv3'], tile_h=8, tile_w=8, tile_oc=8)
    print("Conv3 output shape:", x.shape)  # Debugging output
    x = leaky_relu_loop(x)

    # Conv4 + Leaky ReLU
    x = driver.run_conv_2d(x, weights['conv4'], tile_h=8, tile_w=8, tile_oc=8)
    print("Conv4 output shape:", x.shape)  # Debugging output
    x = leaky_relu_loop(x)

    # 최대 풀링
    x = maxpool_loop(x, pool_size=2, stride=2)
    print("Maxpool output shape:", x.shape)  # Debugging output

    input_reshaped, weight_reshaped = reshape_input_and_weights(x, weights)

    # FC
    output = driver.run_fc_2d(input_reshaped, weight_reshaped, tile_h=3, tile_w=3, tile_oc=1)

    return output.reshape(-1)

In [28]:
start_time = time.time()
#############################################################
output = npu_golden_model_HW(input_image[0], weights)
#############################################################
end_time = time.time()
runtime = end_time - start_time

print("출력 형태:", output.shape)       # (10,) 이어야 함
print("출력 값:", output)               # 출력 값 확인
print(f"Runtime: {runtime*1000:.3f}ms")

Conv1 output shape: (8, 26, 26)
Conv2 output shape: (16, 24, 24)
Conv3 output shape: (64, 22, 22)
Conv4 output shape: (128, 20, 20)
Maxpool output shape: (128, 10, 10)
출력 형태: (10,)
출력 값: [ -53.  -45.   -8.   53. -102.  -51. -128.  127.   14.   14.]
Runtime: 98501.521ms


In [29]:
# start_time = time.time()
# #############################################################
# output = npu_golden_model_HW(input_image[0], weights)
# #############################################################
# end_time = time.time()
# runtime = end_time - start_time

# print("출력 형태:", output.shape)       # (10,) 이어야 함
# print("출력 값:", output)               # 출력 값 확인
# print(f"Runtime: {runtime*1000:.3f}ms")

In [ ]:
### TEST ###
input = np.load('i_act_tile_2.npy')
weight = np.load('weight_tile_2.npy')

driver = NPUDriver()

output = driver.run_conv_2d(input, weight, tile_h=8, tile_w=8, tile_oc=8)
print("출력 형태:", output.shape)       # (8,8,4) 이어야 함
print("출력 값:", output) 